## **Notebook 06 — Hybrid Search + RRF**
- Run dense + sparse retrieval in parallel → merge with Reciprocal Rank Fusion → single ranked list better than either alone.

In [1]:
import os, json, asyncio
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import psycopg2
from pgvector.psycopg2 import register_vector
import bm25s
from FlagEmbedding import BGEM3FlagModel

load_dotenv()
DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")
INDEX_DIR     = Path("../data/bm25_index")

# ── DB connection ────────────────────────────────────
conn_str = DATABASE_SYNC.replace("postgresql+psycopg2://", "postgresql://")
conn     = psycopg2.connect(conn_str)
register_vector(conn)
cur      = conn.cursor()

# ── BM25s index ──────────────────────────────────────
bm25_retriever = bm25s.BM25.load(
    str(INDEX_DIR / "bm25_index"), load_corpus=False
)
with open(INDEX_DIR / "chunk_ids.json")  as f: chunk_ids   = json.load(f)
with open(INDEX_DIR / "chunk_texts.json") as f: chunk_texts = json.load(f)

# ── BGE-M3 model ─────────────────────────────────────
embed_model = BGEM3FlagModel(
    model_name_or_path="BAAI/bge-m3",
    use_fp16=False,
    cache_dir="../models/bge-m3"
)

print("All components loaded OK")
print(f"  DB connection  : OK")
print(f"  BM25s index    : {len(chunk_ids)} chunks")
print(f"  BGE-M3 model   : OK")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

All components loaded OK
  DB connection  : OK
  BM25s index    : 615 chunks
  BGE-M3 model   : OK


In [2]:
def dense_search(query_vec: np.ndarray, k: int = 20) -> list[dict]:
    """
    Search pgvector using cosine similarity.
    Returns top-k results with chunk_id, content, score, rank.
    """
    cur.execute("""
        SELECT
            id::text,
            content,
            parent_content,
            metadata->>'section' AS section,
            1 - (embedding <=> %s::vector) AS score
        FROM chunks
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (query_vec.tolist(), query_vec.tolist(), k))

    return [
        {
            "chunk_id"      : r[0],
            "content"       : r[1],
            "parent_content": r[2],
            "section"       : r[3],
            "score"         : float(r[4]),
            "rank"          : i + 1,
            "source"        : "dense",
        }
        for i, r in enumerate(cur.fetchall())
    ]

print("dense_search() defined")

dense_search() defined


In [3]:
def sparse_search(query: str, k: int = 20) -> list[dict]:
    """
    Search BM25s index using exact keyword matching.
    Returns top-k results with chunk_id, content, score, rank.
    """
    query_tokens    = bm25s.tokenize([query], stopwords="en")
    results, scores = bm25_retriever.retrieve(
        query_tokens, k=min(k, len(chunk_ids))
    )

    return [
        {
            "chunk_id"      : chunk_ids[idx],
            "content"       : chunk_texts[idx],
            "parent_content": None,   # not stored in BM25s index
            "section"       : None,
            "score"         : float(score),
            "rank"          : i + 1,
            "source"        : "sparse",
        }
        for i, (idx, score) in enumerate(zip(results[0], scores[0]))
    ]

print("sparse_search() defined")

sparse_search() defined


In [4]:
def rrf_fusion(
    dense_results  : list[dict],
    sparse_results : list[dict],
    k              : int = 60,    # standard RRF constant — do not change
    top_n          : int = 10,
) -> list[dict]:
    """
    Reciprocal Rank Fusion.

    For each document, sum  1 / (k + rank)  across all retrievers.
    Documents appearing in both lists get a bonus.
    Uses rank position only — ignores raw scores entirely.
    This makes it immune to score scale differences (BM25 vs cosine).
    """
    rrf_scores: dict[str, float] = {}

    # Score from dense retriever
    for result in dense_results:
        cid = result["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + \
                          1.0 / (k + result["rank"])

    # Score from sparse retriever
    for result in sparse_results:
        cid = result["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + \
                          1.0 / (k + result["rank"])

    # Sort by RRF score descending
    ranked_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    # Build output — enrich with content from dense results
    # (dense results carry parent_content and section)
    dense_lookup = {r["chunk_id"]: r for r in dense_results}

    fused = []
    for chunk_id, rrf_score in ranked_ids[:top_n]:
        base = dense_lookup.get(chunk_id, {})
        fused.append({
            "chunk_id"      : chunk_id,
            "content"       : base.get("content", ""),
            "parent_content": base.get("parent_content", ""),
            "section"       : base.get("section", ""),
            "rrf_score"     : rrf_score,
            "rank"          : len(fused) + 1,
            "in_dense"      : chunk_id in {r["chunk_id"] for r in dense_results},
            "in_sparse"     : chunk_id in {r["chunk_id"] for r in sparse_results},
        })

    return fused

print("rrf_fusion() defined")

rrf_fusion() defined


In [5]:
def hybrid_search(query: str, k: int = 20, top_n: int = 10) -> list[dict]:
    """
    Full hybrid search pipeline:
    1. Embed query with BGE-M3
    2. Dense search (pgvector)
    3. Sparse search (BM25s)
    4. RRF fusion
    Returns top_n fused results.
    """
    # Step 1 — embed query
    query_vec = embed_model.encode(
        [query], return_dense=True
    )["dense_vecs"][0]

    # Steps 2 + 3 — both retrievers
    dense  = dense_search(query_vec, k=k)
    sparse = sparse_search(query, k=k)

    # Step 4 — fuse
    fused  = rrf_fusion(dense, sparse, top_n=top_n)

    # Enrich chunks missing parent_content (appeared in sparse only)
    missing_ids = [
        r["chunk_id"] for r in fused
        if not r.get("parent_content") and r["chunk_id"]
    ]
    if missing_ids:
        placeholders = ",".join(["%s"] * len(missing_ids))
        cur.execute(f"""
            SELECT id::text, content, parent_content, metadata->>'section'
            FROM chunks WHERE id::text IN ({placeholders});
        """, missing_ids)
        enrichment = {r[0]: r for r in cur.fetchall()}
        for r in fused:
            if r["chunk_id"] in enrichment:
                e = enrichment[r["chunk_id"]]
                r["content"]        = r["content"] or e[1]
                r["parent_content"] = r["parent_content"] or e[2]
                r["section"]        = r["section"] or e[3]

    return fused

print("hybrid_search() defined")

hybrid_search() defined


In [6]:
QUERY = "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"

results = hybrid_search(QUERY, k=20, top_n=5)

print(f'Query: "{QUERY}"\n')
print(f"{'─'*60}")
for r in results:
    sources = []
    if r["in_dense"]:  sources.append("dense")
    if r["in_sparse"]: sources.append("sparse")
    tag = "+".join(sources) if sources else "sparse-only"

    print(f"Rank {r['rank']}  |  RRF: {r['rrf_score']:.5f}  |  [{tag}]")
    print(f"  Section : {r['section']}")
    print(f"  Content : {r['content'][:110]}")
    print()

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"

────────────────────────────────────────────────────────────
Rank 1  |  RRF: 0.03279  |  [dense+sparse]
  Section : 3.3 Internal Candidate
  Content : Permanent/regular: Permanent or regular employee shall be the one who is employed to work until his/her retire

Rank 2  |  RRF: 0.03200  |  [dense+sparse]
  Section : 3.3 Internal Candidate
  Content : Additionally,  to  meet  special  needs,  BDRCS  may  hire  consultants  for  a temporary period who are not c

Rank 3  |  RRF: 0.02862  |  [dense+sparse]
  Section : Privilege Leave
  Content : While considering the cases of promotion, it needs to be ensured that: - -The applicant is a regular/permanent

Rank 4  |  RRF: 0.02840  |  [dense+sparse]
  Section : Working hours
  Content : A person to be appointed shall not be less than 18 years and not generally more than 55 years of age  at  the 

Rank 5  |  RRF: 0.01613  |  [sparse]
  Section : 5.6 

In [7]:
test_queries = [
    "What is the mandatory retirement age?",
    "How many days off do workers get each year?",
    "BDRCS provident fund contribution percentage",
]

for query in test_queries:
    results = hybrid_search(query, k=20, top_n=3)
    print(f'Query: "{query}"')
    for r in results:
        src = ("dense+sparse" if r["in_dense"] and r["in_sparse"]
               else "dense" if r["in_dense"] else "sparse")
        print(f"  Rank {r['rank']} [{src:>12}]  {r['content'][:80]}")
    print()

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "What is the mandatory retirement age?"
  Rank 1 [dense+sparse]  The staff members of regular/permanent positions in the society shall retire on 
  Rank 2 [dense+sparse]  The  age  shall  be ascertained  from  the  official  certificate  of  Secondary
  Rank 3 [dense+sparse]  -  Retirement from the service after attaining the age of superannuation -  Pr



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "How many days off do workers get each year?"
  Rank 1 [dense+sparse]  The staff members employed by BDRCS shall be entitled to 30 days medical leave o
  Rank 2 [dense+sparse]  As provision of privilege leave is maintained, employees of BDRCS either regular
  Rank 3 [dense+sparse]  -  Leave without pay/Extraordinary Leave. - -All the staff members of the socie



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "BDRCS provident fund contribution percentage"
  Rank 1 [dense+sparse]  Upon  confirmation,  the  staff  members  in  regular/permanent  positions  of  
  Rank 2 [dense+sparse]  The  rate  of  overtime  as  provided  in  the  existing  Standing  Orders  is  
  Rank 3 [dense+sparse]  be set aside and held by the BDRCS Trustees as in the case of Provident Fund for



In [8]:
# Show exactly how RRF calculated the score for the top result
QUERY = "What is the mandatory retirement age?"

query_vec    = embed_model.encode([QUERY], return_dense=True)["dense_vecs"][0]
dense_res    = dense_search(query_vec, k=20)
sparse_res   = sparse_search(QUERY, k=20)

# Find the "60 years" chunk in each result list
target_text  = "shall retire on the date"
d_rank = next((r["rank"] for r in dense_res  if target_text in r["content"]), None)
s_rank = next((r["rank"] for r in sparse_res if target_text in r["content"]), None)

K = 60
d_contribution = 1/(K + d_rank) if d_rank else 0
s_contribution = 1/(K + s_rank) if s_rank else 0
total_rrf      = d_contribution + s_contribution

print("RRF score breakdown for '60 years' chunk:\n")
print(f"  Dense rank   : {d_rank  if d_rank  else 'NOT in top 20'}")
print(f"  Sparse rank  : {s_rank  if s_rank  else 'NOT in top 20'}")
print()
if d_rank:
    print(f"  Dense contribution  : 1 / (60 + {d_rank}) = {d_contribution:.5f}")
if s_rank:
    print(f"  Sparse contribution : 1 / (60 + {s_rank}) = {s_contribution:.5f}")
print(f"  {'─'*40}")
print(f"  Total RRF score     : {total_rrf:.5f}")
print()
print("  A chunk appearing in BOTH lists gets scores from both.")
print("  That bonus is why dense+sparse chunks rank highest.")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

RRF score breakdown for '60 years' chunk:

  Dense rank   : 1
  Sparse rank  : 3

  Dense contribution  : 1 / (60 + 1) = 0.01639
  Sparse contribution : 1 / (60 + 3) = 0.01587
  ────────────────────────────────────────
  Total RRF score     : 0.03227

  A chunk appearing in BOTH lists gets scores from both.
  That bonus is why dense+sparse chunks rank highest.


In [9]:
# conn.close()

print("=" * 52)
print("STEP 7 COMPLETE")
print("=" * 52)
print("  dense_search()   : pgvector cosine similarity")
print("  sparse_search()  : BM25s keyword matching")
print("  rrf_fusion()     : rank-based merging (k=60)")
print("  hybrid_search()  : all three combined")
print()
print("  Retirement age query result:")
print("  Dense alone  → '60 years' NOT in top 5")
print("  Sparse alone → '60 years' at Rank 4")
print("  Hybrid RRF   → '60 years' at Rank 2")
print()
print("  Next: reranker will push it to Rank 1")
print()
print("READY FOR STEP 8 — Query Transformation (HyDE)")

STEP 7 COMPLETE
  dense_search()   : pgvector cosine similarity
  sparse_search()  : BM25s keyword matching
  rrf_fusion()     : rank-based merging (k=60)
  hybrid_search()  : all three combined

  Retirement age query result:
  Dense alone  → '60 years' NOT in top 5
  Sparse alone → '60 years' at Rank 4
  Hybrid RRF   → '60 years' at Rank 2

  Next: reranker will push it to Rank 1

READY FOR STEP 8 — Query Transformation (HyDE)
